# BB84 QKD Simulator — Results & Analysis
**University of Ruhuna — Dept. of Computer Engineering**

This notebook reproduces all figures and tables for the research report.
All experiments are run across three seeds (42, 123, 7). Tables report mean ± std.

| Figure | Experiment | Key insight |
|--------|-----------|-------------|
| A  | QBER vs qubit count n | CI narrows as n grows |
| A2 | CI width vs n | Theoretical 2z/√n scaling |
| B  | QBER vs Eve intercept (multi-n) | Linear QBER–p; small n fails |
| C  | Sample fraction effect | Optimal fraction trade-off |
| D/E | Depolar × Eve heatmap + line plot | Noise masks Eve below 5% |
| F  | Detection probability vs n | Reliability threshold at n≈600 |

## Cell 0 — Imports & Setup

In [1]:
import importlib, sys, warnings, os
import numpy as np
warnings.filterwarnings('ignore')

# Fresh reload
for mod in list(sys.modules.keys()):
    if 'bb84' in mod:
        sys.modules.pop(mod)
importlib.invalidate_caches()

from bb84_config import SimulationConfig
from bb84_runner import run_simulation
from bb84_plots_extended import (
    collect_sample_size_data,
    collect_intercept_sweep_data,
    collect_sample_fraction_data,
    collect_depolar_eve_data,
    plot_sample_size_sensitivity,
    plot_qber_vs_intercept_enhanced,
    plot_sample_fraction_effect,
    plot_depolar_vs_eve_heatmap,
    plot_depolar_vs_eve_lines,
    plot_detection_power,
    collect_detection_power_data,
    plot_ci_width_vs_n,
)

os.makedirs('images', exist_ok=True)

SEEDS = [42, 123, 7]

print('[✓] All imports OK')
print(f'[✓] Images will be saved to: images/images/')
print(f'[✓] Seeds to use: {SEEDS}')

[✓] All imports OK
[✓] Images will be saved to: images/images/
[✓] Seeds to use: [42, 123, 7]


---
## Figure A — Effect of Qubit Count on QBER Estimation
**Research question**: How many qubits are needed for stable QBER estimation?

n = 10, 50, 100, 300, 600, 1000 under ideal conditions and Eve-100%.
Expected: CI width ∝ 1/√(n × sample_fraction). Multi-seed: each configuration run with seeds 42, 123, 7.

In [3]:
N_VALUES = [10, 50, 100, 300, 600, 1000]

data_A_ideal_seeds  = []
data_A_eve100_seeds = []

for seed in SEEDS:
    print(f'\n=== Seed {seed}: Ideal channel (no Eve) ===')
    data_A_ideal_seeds.append(
        collect_sample_size_data(N_VALUES, eve_intercept=0.0,
                                  sample_fraction=0.15, seed=seed)
    )
    print(f'\n=== Seed {seed}: Eve full intercept (100%) ===')
    data_A_eve100_seeds.append(
        collect_sample_size_data(N_VALUES, eve_intercept=1.0,
                                  sample_fraction=0.15, seed=seed)
    )

print('\n[✓] Figure A data collection complete.')


=== Seed 42: Ideal channel (no Eve) ===
    n=   10  QBER=0.0%  CI=[0.0,79.3]  status=SECURE ok
    n=   50  QBER=0.0%  CI=[0.0,56.2]  status=SECURE ok
    n=  100  QBER=0.0%  CI=[0.0,35.4]  status=SECURE ok
    n=  300  QBER=0.0%  CI=[0.0,14.9]  status=SECURE ok
    n=  600  QBER=0.0%  CI=[0.0,8.0]  status=SECURE ok
    n= 1000  QBER=0.0%  CI=[0.0,4.8]  status=SECURE ok

=== Seed 42: Eve full intercept (100%) ===
    n=   10  QBER=0.0%  CI=[0.0,79.3]  status=SECURE ok
    n=   50  QBER=66.7%  CI=[20.8,93.9]  status=ABORT x
    n=  100  QBER=14.3%  CI=[2.6,51.3]  status=ABORT x
    n=  300  QBER=18.2%  CI=[7.3,38.5]  status=ABORT x
    n=  600  QBER=20.5%  CI=[11.2,34.5]  status=ABORT x
    n= 1000  QBER=23.7%  CI=[15.5,34.4]  status=ABORT x

=== Seed 123: Ideal channel (no Eve) ===
    n=   10  QBER=0.0%  CI=[0.0,79.3]  status=SECURE ok
    n=   50  QBER=0.0%  CI=[0.0,56.2]  status=SECURE ok
    n=  100  QBER=0.0%  CI=[0.0,35.4]  status=SECURE ok
    n=  300  QBER=0.0%  CI=[0.0,14.9]

In [ ]:
# ── A-2: PLOT (using seed=42 data for representative plots) ─────────
plot_sample_size_sensitivity(
    data_ideal  = data_A_ideal_seeds[0],
    data_eve100 = data_A_eve100_seeds[0],
    n_values    = N_VALUES,
    save_path   = 'images/fig_A_sample_size.png',
)

plot_ci_width_vs_n(
    data_ideal       = data_A_ideal_seeds[0],
    data_eve100      = data_A_eve100_seeds[0],
    n_values         = N_VALUES,
    sample_fraction  = 0.15,
    save_path        = 'images/fig_A2_ci_width.png',
)

In [ ]:
# ── Table A — Mean ± Std across 3 seeds ─────────────────────────────
print('Table A-1: Ideal Channel — QBER (%) across seeds 42, 123, 7')
print('=' * 85)
print(f"{'n':>6}  {'Seed 42':>9}  {'Seed 123':>9}  {'Seed 7':>9}  {'Mean':>8}  {'Std':>8}")
print('-' * 85)
for ni, n in enumerate(N_VALUES):
    vals = [data_A_ideal_seeds[si][ni]['qber'] for si in range(len(SEEDS))]
    print(f"{n:>6}  {vals[0]:>8.1f}%  {vals[1]:>8.1f}%  {vals[2]:>8.1f}%  "
          f"{np.mean(vals):>7.1f}%  {np.std(vals):>7.2f}%")

print()
print('Table A-2: Eve 100% — QBER (%) across seeds 42, 123, 7')
print('=' * 85)
print(f"{'n':>6}  {'Seed 42':>9}  {'Seed 123':>9}  {'Seed 7':>9}  {'Mean':>8}  {'Std':>8}")
print('-' * 85)
for ni, n in enumerate(N_VALUES):
    vals = [data_A_eve100_seeds[si][ni]['qber'] for si in range(len(SEEDS))]
    print(f"{n:>6}  {vals[0]:>8.1f}%  {vals[1]:>8.1f}%  {vals[2]:>8.1f}%  "
          f"{np.mean(vals):>7.1f}%  {np.std(vals):>7.2f}%")

print()
print('Table A-3: Eve 100% — 95% CI Width (%) across seeds 42, 123, 7')
print('=' * 85)
print(f"{'n':>6}  {'Seed 42':>9}  {'Seed 123':>9}  {'Seed 7':>9}  {'Mean':>8}  {'Std':>8}")
print('-' * 85)
for ni, n in enumerate(N_VALUES):
    vals = [data_A_eve100_seeds[si][ni]['ci_width'] for si in range(len(SEEDS))]
    print(f"{n:>6}  {vals[0]:>8.1f}%  {vals[1]:>8.1f}%  {vals[2]:>8.1f}%  "
          f"{np.mean(vals):>7.1f}%  {np.std(vals):>7.2f}%")

---
## Figure B — QBER vs. Eve Intercept Probability (Multi-n)
Sweep p_Eve: 0% → 100% (11 steps) for n = 40, 100, 400, 600.
Theory: QBER = 0.25 × p_Eve.

In [4]:
N_LIST_B = [40, 100, 400, 600]
STEPS_B  = 10

sweep_data_B_seeds = []

for seed in SEEDS:
    print(f'\n========== Seed {seed} ==========')
    sd = {}
    for n in N_LIST_B:
        print(f'\n--- n = {n} ---')
        sd[n] = collect_intercept_sweep_data(
            n_qubits=n, steps=STEPS_B,
            sample_fraction=0.15, seed=seed
        )
    sweep_data_B_seeds.append(sd)

print('\n[✓] Figure B data collection complete.')


========== Seed 42 ==========

--- n = 40 ---
    p=0.00  QBER=0.0%
    p=0.10  QBER=0.0%
    p=0.20  QBER=0.0%
    p=0.30  QBER=33.3%
    p=0.40  QBER=33.3%
    p=0.50  QBER=0.0%
    p=0.60  QBER=0.0%
    p=0.70  QBER=0.0%
    p=0.80  QBER=0.0%
    p=0.90  QBER=66.7%
    p=1.00  QBER=66.7%

--- n = 100 ---
    p=0.00  QBER=0.0%
    p=0.10  QBER=0.0%
    p=0.20  QBER=0.0%
    p=0.30  QBER=0.0%
    p=0.40  QBER=0.0%
    p=0.50  QBER=0.0%
    p=0.60  QBER=0.0%
    p=0.70  QBER=0.0%
    p=0.80  QBER=0.0%
    p=0.90  QBER=0.0%
    p=1.00  QBER=14.3%

--- n = 400 ---
    p=0.00  QBER=0.0%
    p=0.10  QBER=3.2%
    p=0.20  QBER=6.5%
    p=0.30  QBER=3.2%
    p=0.40  QBER=12.9%
    p=0.50  QBER=6.5%
    p=0.60  QBER=9.7%
    p=0.70  QBER=12.9%
    p=0.80  QBER=12.9%
    p=0.90  QBER=16.1%
    p=1.00  QBER=16.1%

--- n = 600 ---
    p=0.00  QBER=0.0%
    p=0.10  QBER=2.3%
    p=0.20  QBER=4.5%
    p=0.30  QBER=6.8%
    p=0.40  QBER=9.1%
    p=0.50  QBER=11.4%
    p=0.60  QBER=13.6%
    p=0.70

In [ ]:
# ── B-2: PLOT (seed=42 as representative) ────────────────────────────
plot_qber_vs_intercept_enhanced(
    sweep_data = sweep_data_B_seeds[0],
    n_list     = N_LIST_B,
    save_path  = 'images/fig_B_qber_vs_intercept.png',
)

In [ ]:
# ── Table B — Observed QBER (%) as a function of p_Eve, Mean ± Std ──
p_eve_labels = [f'{p:.1f}' for p in np.linspace(0, 1, STEPS_B + 1)]

print('Table B: Observed QBER (%) — Mean ± Std across seeds 42, 123, 7')
print('Experiment: Intercept-resend attack, f=0.15')
print('=' * 95)

header = f"{'p_Eve':>6}  "
for n in N_LIST_B:
    header += f"{'n='+str(n)+' (Mean±Std)':>22}  "
print(header)
print('-' * 95)

for pi, p_label in enumerate(p_eve_labels):
    row = f"{p_label:>6}  "
    for n in N_LIST_B:
        qbers_across_seeds = [
            sweep_data_B_seeds[si][n][1][pi]
            for si in range(len(SEEDS))
        ]
        mean_q = np.mean(qbers_across_seeds)
        std_q  = np.std(qbers_across_seeds)
        row += f"{mean_q:>7.1f} ± {std_q:>5.1f}     "
    print(row)

print('=' * 95)
print('Note: Theory predicts QBER = 25 × p_Eve %.')

---
## Figure C — Effect of Sample Fraction on QBER Estimation
Sample fractions: 5%, 10%, 15%, 20%, 30% at Eve rates p = 0.3, 0.5, 1.0 (n=600).

In [ ]:
INTERCEPT_RATES_C = [0.3, 0.5, 1.0]
FRAC_VALUES_C     = [0.05, 0.10, 0.15, 0.20, 0.30]

frac_data_C_seeds = []
for seed in SEEDS:
    print(f'\n=== Seed {seed}: Sample fraction sweep (n=600) ===')
    frac_data_C_seeds.append(
        collect_sample_fraction_data(
            intercept_rates=INTERCEPT_RATES_C,
            frac_values=FRAC_VALUES_C,
            n_qubits=600,
            seed=seed,
        )
    )

print('\n[✓] Figure C (n=600) data collection complete.')

In [ ]:
# ── C-2: PLOT (seed=42 representative) ──────────────────────────────
plot_sample_fraction_effect(
    frac_data       = frac_data_C_seeds[0],
    intercept_rates = INTERCEPT_RATES_C,
    frac_values     = FRAC_VALUES_C,
    save_path       = 'images/fig_C600_sample_fraction.png',
)

In [ ]:
# ── Table C — CI width at each fraction (Eve=100%, n=600), Mean ± Std
print('Table C: CI Width at Each Sample Fraction — Eve=100%, n=600')
print('Mean ± Std across seeds 42, 123, 7')
print('=' * 90)
print(f"{'Fraction':>10}  {'Seed 42 QBER':>13}  {'Seed 123 QBER':>14}  "
      f"{'Seed 7 QBER':>12}  {'Mean QBER':>11}  {'Std':>7}  {'Mean CI-W':>10}  Status")
print('-' * 90)

for frac in FRAC_VALUES_C:
    qbers = [frac_data_C_seeds[si][1.0][frac]['qber'] for si in range(len(SEEDS))]
    ci_ws = [
        frac_data_C_seeds[si][1.0][frac]['ci_high'] -
        frac_data_C_seeds[si][1.0][frac]['ci_low']
        for si in range(len(SEEDS))
    ]
    status = frac_data_C_seeds[0][1.0][frac]['status']
    print(f"  {frac*100:>6.0f}%  {qbers[0]:>12.1f}%  {qbers[1]:>13.1f}%  "
          f"{qbers[2]:>11.1f}%  {np.mean(qbers):>10.1f}%  "
          f"{np.std(qbers):>6.2f}%  {np.mean(ci_ws):>9.1f}%  {status}")

---
## Figure D & E — Channel Noise × Eve Intercept
Grid: depolar p ∈ {0, 0.01, 0.03, 0.05} × Eve p ∈ {0, 0.2, 0.4, 0.6, 0.8, 1.0} (n=600).

In [2]:
DEPOLAR_VALUES = [0.000, 0.010, 0.030, 0.050]
EVE_VALUES     = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

qber_matrices_DE_seeds = []
for seed in SEEDS:
    print(f'\n=== Seed {seed}: Depolar × Eve grid (n=600) ===')
    qber_matrices_DE_seeds.append(
        collect_depolar_eve_data(
            depolar_values  = DEPOLAR_VALUES,
            eve_values      = EVE_VALUES,
            n_qubits        = 600,
            sample_fraction = 0.15,
            seed            = seed,
        )
    )

qber_matrix_DE_mean = np.mean(qber_matrices_DE_seeds, axis=0)
qber_matrix_DE_std  = np.std(qber_matrices_DE_seeds,  axis=0)

print('\n[✓] Figure D/E data collection complete.')


=== Seed 42: Depolar × Eve grid (n=600) ===
    dp=0.000  ep=0.00  QBER=0.0%
    dp=0.000  ep=0.20  QBER=4.5%
    dp=0.000  ep=0.40  QBER=9.1%
    dp=0.000  ep=0.60  QBER=13.6%
    dp=0.000  ep=0.80  QBER=13.6%
    dp=0.000  ep=1.00  QBER=20.5%
    dp=0.010  ep=0.00  QBER=0.0%
    dp=0.010  ep=0.20  QBER=6.8%
    dp=0.010  ep=0.40  QBER=15.9%
    dp=0.010  ep=0.60  QBER=20.5%
    dp=0.010  ep=0.80  QBER=20.5%
    dp=0.010  ep=1.00  QBER=25.0%
    dp=0.030  ep=0.00  QBER=2.3%
    dp=0.030  ep=0.20  QBER=9.1%
    dp=0.030  ep=0.40  QBER=18.2%
    dp=0.030  ep=0.60  QBER=22.7%
    dp=0.030  ep=0.80  QBER=22.7%
    dp=0.030  ep=1.00  QBER=27.3%
    dp=0.050  ep=0.00  QBER=4.5%
    dp=0.050  ep=0.20  QBER=9.1%
    dp=0.050  ep=0.40  QBER=18.2%
    dp=0.050  ep=0.60  QBER=20.5%
    dp=0.050  ep=0.80  QBER=22.7%
    dp=0.050  ep=1.00  QBER=25.0%

=== Seed 123: Depolar × Eve grid (n=600) ===
    dp=0.000  ep=0.00  QBER=0.0%
    dp=0.000  ep=0.20  QBER=7.1%
    dp=0.000  ep=0.40  QBER=9.5%
   

In [ ]:
# ── D-2: HEATMAP ─────────────────────────────────────────────────────
plot_depolar_vs_eve_heatmap(
    qber_matrix    = qber_matrix_DE_mean,
    depolar_values = DEPOLAR_VALUES,
    eve_values     = EVE_VALUES,
    save_path      = 'images/fig_D_depolar_heatmap.png',
)

# ── E-2: LINE PLOT ───────────────────────────────────────────────────
plot_depolar_vs_eve_lines(
    qber_matrix    = qber_matrix_DE_mean,
    depolar_values = DEPOLAR_VALUES,
    eve_values     = EVE_VALUES,
    save_path      = 'images/fig_E_depolar_lines.png',
)

In [ ]:
# ── Table D/E — Raw QBER matrix: Mean ± Std across seeds ─────────────
print('Table D/E: QBER (%) as a function of Depolarising Noise and Eve Intercept')
print('Mean ± Std across seeds 42, 123, 7  |  n=600, f=0.15')
print('=' * 80)

eve_hdr = '   '.join([f'Eve {int(e*100)}%' for e in EVE_VALUES])
print(f"{'p_dep':>8}   {eve_hdr}")
print('-' * 80)

for i, dp in enumerate(DEPOLAR_VALUES):
    row_parts = []
    for j in range(len(EVE_VALUES)):
        mean_v = qber_matrix_DE_mean[i, j]
        std_v  = qber_matrix_DE_std[i, j]
        row_parts.append(f"{mean_v:5.1f}±{std_v:4.1f}")
    row = '  '.join(row_parts)
    print(f"  {dp:.3f}   {row}")

---
## Figure F — Eve Detection Probability vs n
20 trials per n value. At what qubit count does the protocol reliably detect Eve?

In [ ]:
N_VALUES_F = [50, 100, 200, 300, 500, 700, 1000]
N_TRIALS_F = 20

print('=== Detection Power (Eve=100%, f=0.15, 20 trials per n) ===')
detect_probs_F = collect_detection_power_data(
    n_values      = N_VALUES_F,
    eve_intercept = 1.0,
    n_trials      = N_TRIALS_F,
    sample_frac   = 0.15,
)

print('\n[✓] Figure F data collection complete.')

In [ ]:
plot_detection_power(
    n_values     = N_VALUES_F,
    detect_probs = detect_probs_F,
    n_trials     = N_TRIALS_F,
    sample_frac  = 0.15,
    save_path    = 'images/fig_F_detection_power.png',
)

In [ ]:
# ── Table F — Detection Probability per n ────────────────────────────
print('Table F: Eve Detection Probability vs. Qubit Count')
print(f'Eve intercept=100%, f=0.15, {N_TRIALS_F} trials, ABORT threshold = QBER ≥ 11%')
print('=' * 50)
print(f"{'n':>8}  {'Detection Prob (%)':>20}  {'Verdict':>12}")
print('-' * 50)
for n, dp in zip(N_VALUES_F, detect_probs_F):
    verdict = 'Reliable' if dp >= 95 else ('Moderate' if dp >= 60 else 'Unreliable')
    print(f"{n:>8}  {dp:>19.1f}%  {verdict:>12}")
print('=' * 50)

## Summary: All Images Saved

In [ ]:
import os

print('Images saved to images/ folder:')
print('=' * 50)
saved = sorted([f for f in os.listdir('images') if f.endswith('.png')])
for f in saved:
    size_kb = os.path.getsize(f'images/{f}') / 1024
    print(f'  images/{f}  ({size_kb:.0f} KB)')
print(f'\nTotal: {len(saved)} image(s)')